In [5]:
import pandas as pd
import pandas_gbq
from pathlib import Path

PROJECT_ID = "sylvan-agility-511910-s9"
DATASET = "funnel_project"

raw = Path("../data/raw")
raw.mkdir(parents=True, exist_ok=True)

test = pandas_gbq.read_gbq(
    f"SELECT * FROM `{PROJECT_ID}.{DATASET}.orders_table` LIMIT 100", project_id=PROJECT_ID
)
test

c:\projects\product-funnel-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


,transaction_id,session_id,session_key,full_visitor_id,order_date,order_ts,hit_number,revenue_usd,tax_usd,shipping_usd,currency
0,ORD20160801563,2788555723243018631-1470077433,2788555723243018631-1470077433-20160801,2788555723243018631,2016-08-01,2016-08-01 19:09:17.567000+00:00,63,258.00,NaN,NaN,USD
1,ORD201608121550,0421965906840211099-1471025250,0421965906840211099-1471025250-20160812,0421965906840211099,2016-08-12,2016-08-12 18:11:51.200000+00:00,15,109.00,NaN,NaN,USD
2,ORD201608152162,2934242886567182756-1471309860,2934242886567182756-1471309860-20160815,2934242886567182756,2016-08-15,2016-08-16 01:22:56.509000+00:00,25,27.00,NaN,NaN,USD
3,ORD201608191490,7308573211245935680-1471616624,7308573211245935680-1471616624-20160819,7308573211245935680,2016-08-19,2016-08-19 14:43:37.344000+00:00,22,59.00,NaN,NaN,USD
4,ORD201608241571,3656215501791704148-1472061530,3656215501791704148-1472061530-20160824,3656215501791704148,2016-08-24,2016-08-24 18:05:13.195000+00:00,8,31.00,6.00,NaN,USD
...,...,...,...,...,...,...,...,...,...,...,...
95,ORD201704211956,2606909581831356927-1492779366,2606909581831356927-1492779366-20170421,2606909581831356927,2017-04-21,2017-04-21 13:33:37.773000+00:00,16,20.42,3.56,7.0,USD
96,ORD201704211994,6428878966962078899-1492789222,6428878966962078899-1492789222-20170421,6428878966962078899,2017-04-21,2017-04-21 15:50:28.453000+00:00,19,16.19,2.49,7.0,USD
97,ORD201704221181,8889391457640720040-1492858313,8889391457640720040-1492858313-20170422,8889391457640720040,2017-04-22,2017-04-22 10:58:05.055000+00:00,8,42.97,NaN,7.0,USD
98,ORD201704242099,1346851855808049432-1493063899,1346851855808049432-1493063899-20170424,1346851855808049432,2017-04-24,2017-04-24 20:28:46.288000+00:00,27,86.96,NaN,7.0,USD


In [6]:
tables = ["orders_table","users","sessions","events"]

for t in tables:
    df = pandas_gbq.read_gbq(
        f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{t}`", project_id=PROJECT_ID
    )
    for c in df.columns:
        if str(df[c].dtype) == "dbdate":
            df[c] = pd.to_datetime(df[c])
    print(t, df.shape)
    print(df.dtypes, "\n")
    df.to_parquet(raw / f"{t}.parquet",index=False)
    del df

c:\projects\product-funnel-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


orders_table (11552, 11)
transaction_id                     str
session_id                         str
session_key                        str
full_visitor_id                    str
order_date               datetime64[s]
order_ts           datetime64[us, UTC]
hit_number                       Int64
revenue_usd                    float64
tax_usd                        float64
shipping_usd                   float64
currency                           str
dtype: object 



c:\projects\product-funnel-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


users (714167, 8)
full_visitor_id                 str
first_visit_number            Int64
first_session_date    datetime64[s]
cohort_month          datetime64[s]
is_true_new                 boolean
first_channel                   str
first_category                  str
first_country                   str
dtype: object 



c:\projects\product-funnel-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


sessions (903653, 17)
session_key                            str
session_id                             str
full_visitor_id                        str
visit_id                             Int64
session_date                 datetime64[s]
visit_start_ts         datetime64[us, UTC]
visit_number                         Int64
channel_grouping                       str
device_category                        str
country                                str
pageviews                            Int64
total_hits                           Int64
is_bounce                          boolean
transactions                         Int64
session_revenue_usd                float64
is_split_visit                     boolean
is_continuation_row                boolean
dtype: object 



c:\projects\product-funnel-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2926: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  record_batch = self.to_arrow(


events (1059291, 10)
event_id                           str
session_key                        str
session_id                         str
full_visitor_id                    str
event_ts           datetime64[us, UTC]
hit_number                       Int64
hit_type                           str
action_type                        str
event_name                         str
transaction_id                     str
dtype: object 



In [7]:
sessions = pd.read_parquet(raw / "sessions.parquet")
events = pd.read_parquet(raw / "events.parquet")
orders = pd.read_parquet(raw / "orders_table.parquet")
users = pd.read_parquet(raw / "users.parquet")

checks = {
    "sessions rows":           (len(sessions), 903_653),
    "sessions unique key":     (sessions["session_key"].is_unique, True),
    "sessions revenue":        (round(float(sessions["session_revenue_usd"].sum()), 2), 1_540_071.24),
    "sessions purchase rows":  (int(sessions["transactions"].notna().sum()), 11_552),
    "events rows":             (len(events), 1_059_291),
    "events unique id":        (events["event_id"].is_unique, True),
    "events purchase sessions":(events.loc[events["event_name"] == "purchase", "session_key"].nunique(), 11_552),
    "orders rows":             (len(orders), 11_552),
    "orders unique id":        (orders["transaction_id"].is_unique, True),
    "orders revenue":          (round(float(orders["revenue_usd"].sum()), 2), 1_627_429.54),
    "users rows":              (len(users), 714_167),
    "users true new":          (int(users["is_true_new"].sum()), 701_500),
}

for name, (got, expected) in checks.items():
    ok = abs(got - expected) < 0.01 if isinstance(expected, float) else got == expected
    print(("OK  " if ok else "FAIL ") + f"{name}: got {got}, expected {expected}")

OK  sessions rows: got 903653, expected 903653
OK  sessions unique key: got True, expected True
OK  sessions revenue: got 1540071.24, expected 1540071.24
OK  sessions purchase rows: got 11552, expected 11552
OK  events rows: got 1059291, expected 1059291
OK  events unique id: got True, expected True
OK  events purchase sessions: got 11552, expected 11552
OK  orders rows: got 11552, expected 11552
OK  orders unique id: got True, expected True
OK  orders revenue: got 1627429.54, expected 1627429.54
OK  users rows: got 714167, expected 714167
OK  users true new: got 701500, expected 701500
